# InvoiceAI — Web demo (Gradio)

Starts the InvoiceAI web interface in Colab and gives you a **public link** you can open on any device or send to a client.

**Before you start:** `Runtime → Change runtime type → T4 GPU → Save`.

Do not run this notebook and `04_api.ipynb` at the same time: both load the model, and the T4 does not have memory for two copies.

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Get the code

In [ ]:
GITHUB_USER = "YOUR_GITHUB_USERNAME"  # <- change this
REPO_DIR = "/content/invoice-ai"

import os
if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone https://github.com/{GITHUB_USER}/invoice-ai.git {REPO_DIR}
!ls {REPO_DIR}/ui {REPO_DIR}/samples

## 3. Install requirements

This installs **gradio 5** next to **transformers 4** (Colab comes with gradio 6, which does not work with transformers 4).

A warning about `diffusers` and `huggingface-hub` may appear. It is harmless: we do not use diffusers.

**After this cell: `Runtime → Restart session`**, then continue with step 4 (do not run steps 2–3 again).

In [ ]:
!pip install -q -r /content/invoice-ai/requirements.txt

## 4. Check the versions

This checks that gradio and the Qwen2.5-VL code of transformers work together. All lines should say OK.

In [ ]:
import os, sys
os.chdir("/content/invoice-ai")
sys.path.insert(0, "/content/invoice-ai")

from importlib.metadata import version

versions = {name: version(name) for name in ["transformers", "huggingface-hub", "gradio", "torch"]}
for name, ver in versions.items():
    print(f"{name:16} {ver}")

major = lambda name: int(versions[name].split(".")[0])
assert major("transformers") == 4, "transformers must be 4.x. Run step 3 again and restart the session."
assert major("gradio") == 5, "gradio must be 5.x. Run step 3 again and restart the session."
assert major("huggingface-hub") == 0, "huggingface-hub must be 0.x. Run step 3 again and restart the session."
print("OK: versions")

import gradio as gr
from transformers import Qwen2_5_VLForConditionalGeneration  # noqa: F401
print("OK: gradio and Qwen2.5-VL can be imported together")

## 5. Load the model

Takes 1–3 minutes (the first time it downloads about 7 GB).

In [ ]:
import logging, os, sys, torch
os.chdir("/content/invoice-ai")
sys.path.insert(0, "/content/invoice-ai")
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s", force=True)

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU found. Go to Runtime -> Change runtime type -> T4 GPU. "
        "If Colab gives you no GPU, wait a few hours or use Kaggle."
    )

from app.pipeline import get_pipeline

pipeline = get_pipeline()
pipeline.load()
print("Model loaded on", torch.cuda.get_device_name(0))

## 6. Quick test (without the UI)

Runs one sample file through the pipeline. This proves the model still works after the new install.

In [ ]:
from pathlib import Path
from IPython.display import display

from app.pipeline import SUPPORTED_EXTENSIONS

sample = sorted(p for p in Path("samples").iterdir() if p.suffix.lower() in SUPPORTED_EXTENSIONS)[0]
doc = pipeline.process_file(sample)
total = doc.result.fields["total_amount"]
print(f"File:  {sample.name}")
print(f"Total: {total.value}  (confidence: {total.confidence}, source: {total.source})")
print(f"Time:  {doc.result.processing_seconds} s")

preview = doc.draw(0)
preview.thumbnail((500, 700))
display(preview)

## 7. Start the web app

Wait until you see a line like `Running on public URL: https://xxxx.gradio.live`. Open that link.

- The link works on any device (phone, laptop) while this Colab session is running.
- The first file takes a bit longer than the rest.
- Files are processed one at a time. If two people upload at once, the second waits in a queue.

In [ ]:
from ui.gradio_app import launch_app

demo = launch_app(pipeline, share=True, prevent_thread_lock=True)

## 8. Stop the app

Run this when you are done (or after you change code and want to restart it).
After changing code on GitHub: `git pull` (step 2), **Runtime → Restart session**, then run steps 4, 5 and 7 again.

In [ ]:
demo.close()
print("App stopped.")